# Setup

In [1]:
import pickle
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse import csr_matrix
import implicit
import time
import os

c:\Users\madina\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
import os
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.insert(0, project_root)

from src.custom_iALS import ImplicitALS
from src.als_eval import evaluate_als, evaluate_implicit

In [3]:
with open("../data/artifacts/eval_setup.pkl", "rb") as f:
    setup = pickle.load(f)

In [4]:
val_user_ids = setup["user_ids"]             
gt_padded = setup["gt_padded"]               
gt_sizes = setup["gt_sizes"]     
K = setup["K"]          
discounts = setup["discounts"]  
max_gt = setup["max_gt"]

print(f"val users: {len(val_user_ids):,}")
print(f"gt avg size: {gt_sizes.mean():.2f}")

val users: 91,497
gt avg size: 1.78


In [5]:
df = pd.read_parquet("../data/interactions_final.parquet", columns=["user_id", "parent_asin", "rating", 'split'])
train = df[df["split"] == "train"].copy() 
train.drop(columns='split', inplace=True)

In [6]:
print(f"train shape: {train.shape}")
print(f"unique users: {train['user_id'].nunique():,}")
print(f"unique items: {train['parent_asin'].nunique():,}")

train shape: (3230426, 3)
unique users: 692,133
unique items: 308,122


In [7]:
user_to_idx = {u: i for i, u in enumerate(train["user_id"].unique())}
item_to_idx = {a: i for i, a in enumerate(train["parent_asin"].unique())}
idx_to_user = {i: u for u, i in user_to_idx.items()}
idx_to_item = {i: a for a, i in item_to_idx.items()}
n_users = len(user_to_idx)
n_items = len(item_to_idx)

In [8]:
train["user_idx"] = train["user_id"].map(user_to_idx)
train["item_idx"] = train["parent_asin"].map(item_to_idx)

# Sparce matrices

- Preference: binary (iteraction = 1)
- Confidence: `c = 1 + alpha * (rating - 1)`, so that rating 5 has bigger weight (alpha is a coefficient from article and equals 20)

In [9]:
ratings = train["rating"].values.astype(np.float32)
user_idx_arr = train["user_idx"].values.astype(np.int64)
item_idx_arr = train["item_idx"].values.astype(np.int64)

In [10]:
alpha = 20.0 
preferences = np.ones_like(ratings, dtype=np.float32)
confidence = 1.0 + alpha * (ratings - 1.0) 

In [ ]:
R = csr_matrix((preferences, (user_idx_arr, item_idx_arr)), shape=(n_users, n_items))
C = csr_matrix((confidence, (user_idx_arr, item_idx_arr)), shape=(n_users, n_items))

In [12]:
print(f"R (and C): shape = {R.shape}")
print(f"non-zeros = {R.nnz:,}")
print(f"density = {R.nnz / (n_users * n_items):.8f}")
print(f"sparsity = {1 - R.nnz / (n_users * n_items):.6f}")

R (and C): shape = (692133, 308122)
non-zeros = 3,230,426
density = 0.00001515
sparsity = 0.999985


# Custom ALS on subsample

custom implementation of implicit ALS is in `../src/custom_iALS.py`. as the computations for the whole initial dataset will take a lot of time, here I'm testing the custom version on a subset from train (50k most active users). predictions for all train data will be made via `implicit` library version of ALS

In [13]:
user_activity = train.groupby("user_idx").size().sort_values(ascending=False)
top_users = user_activity.head(50_000).index.values

In [14]:
train_subset = train[train["user_idx"].isin(top_users)].copy()
print(f"Subset: {len(train_subset):,} interactions, {train_subset['user_idx'].nunique():,} users, {train_subset['item_idx'].nunique():,} items")

Subset: 1,178,999 interactions, 50,000 users, 207,076 items


## New sparce matrices on subset

In [15]:
subset_user_indices = np.sort(train_subset["user_idx"].unique())
subset_user_to_local = {u_idx: i for i, u_idx in enumerate(subset_user_indices)}

In [16]:
train_subset["user_idx_local"] = train_subset["user_idx"].map(subset_user_to_local)
subset_n_users = len(subset_user_indices)
subset_n_items = n_items

In [17]:
ss_user_idx_arr = train_subset["user_idx_local"].values
ss_item_idx_arr = train_subset["item_idx"].values
ss_ratings = train_subset["rating"].values

In [18]:
ss_preferences = np.ones_like(ss_ratings)
ss_confidence = 1.0 + alpha * (ss_ratings - 1.0)

In [19]:
R_subset = csr_matrix((ss_preferences, (ss_user_idx_arr, ss_item_idx_arr)), shape=(subset_n_users, subset_n_items))
C_subset = csr_matrix((ss_confidence, (ss_user_idx_arr, ss_item_idx_arr)),shape=(subset_n_users, subset_n_items))

## Fitting and evaluating on val

In [ ]:
custom_als = ImplicitALS(n_factors=32, reg=0.1, alpha=20, n_iter=5)
t0 = time.time()
custom_als.fit(R_subset, C_subset, subset_n_users, subset_n_items)
cust_time = time.time() - t0
print(f"training time: {cust_time:.1f}")

training time: 0.7 min


In [21]:
# predicting only for users in both subset and val
val_user_idx_global = np.array([user_to_idx[u] for u in val_user_ids]) 
in_subset = np.isin(val_user_idx_global, subset_user_indices)
print(f"Val users in subset: {in_subset.sum():,} / {len(val_user_idx_global):,}")

Val users in subset: 15,570 / 91,497


In [22]:
# indices of val users in subset 
val_user_idx_local = np.array([subset_user_to_local[u] for u in val_user_idx_global[in_subset]])
gt_padded_sub = gt_padded[in_subset]
gt_sizes_sub = gt_sizes[in_subset]

In [23]:
als_metrics = evaluate_als(
    model=custom_als,
    user_idx_local=val_user_idx_local,
    gt_padded=gt_padded_sub,
    gt_sizes=gt_sizes_sub,
    K=10, batch_size=2500
)

In [33]:
print("custom ALS (50k subset) on val:")
for k, v in als_metrics.items():
    print(f"{k}: {v:.6f}")

custom ALS (50k subset) on val:
precision@10: 0.004528
recall@10: 0.015496
ndcg@10: 0.009496
predict time (s): 39.558817


ALS (50k subset) on val:
precision@10: 0.004528
recall@10: 0.015496
ndcg@10: 0.009045

custom ALS fitted on 50k users (7% of all users in train) is already beating popularity baseline in all metrics. Results confirm collaborative filtering captures personalization beyond popularity. 

# ALS from implicit library on subset

fitting `implicit.als.AlternatingLeastSquares` on the same subset of 50k to compare our implementation with library version 

## Fitting and evaluating

In [39]:
R_subset_raw = csr_matrix((train_subset['rating'].values, (train_subset['user_idx_local'].values, train_subset['item_idx'].values)),
                          shape=(subset_n_users, subset_n_items))
print(f"R_raw_subset: shape={R_subset_raw.shape}, non-zeros={R_subset_raw.nnz:,}")
print(f"Rating range: [{R_subset_raw.data.min():.1f}, {R_subset_raw.data.max():.1f}]")

R_raw_subset: shape=(50000, 308122), non-zeros=1,178,999
Rating range: [1.0, 5.0]


In [40]:
impl_als_sub = implicit.als.AlternatingLeastSquares(factors=32, regularization=0.1,
                                                alpha=20, iterations=5, random_state=42, num_threads=0)
t0 = time.time()
impl_als_sub.fit(R_subset_raw)
impl_time = time.time() - t0
print(f"training time: {impl_time:.1f}s")

100%|██████████| 5/5 [00:02<00:00,  2.33it/s]

training time: 2.2s


In [41]:
print(f"user_factors: {impl_als_sub.user_factors.shape}")
print(f"item_factors: {impl_als_sub.item_factors.shape}")

user_factors: (50000, 32)
item_factors: (308122, 32)


In [ ]:
impl_metrics = evaluate_implicit(
    model=impl_als_sub,
    user_idx_local=val_user_idx_local,
    R=R_subset_raw,
    gt_padded=gt_padded_sub,
    gt_sizes=gt_sizes_sub,
    K=10,
)

In [43]:
print("implicit ALS (50k subset) on val:")
for k, v in impl_metrics.items():
    print(f"{k}: {v:.6f}")

implicit ALS (50k subset) on val:
precision@10: 0.004496
recall@10: 0.015826
ndcg@10: 0.009521
predict time (s): 10.681962


## Comparing custom and library ALS

In [52]:
sub_df = pd.DataFrame([als_metrics, impl_metrics])
sub_df['ALS'] = ['Custom', 'Implicit lib']
sub_df['fit time (s)'] = [cust_time, impl_time]
sub_df[['ALS','precision@10', 'recall@10', 'ndcg@10', 'fit time (s)','predict time (s)']]

,ALS,precision@10,recall@10,ndcg@10,fit time (s),predict time (s)
0,Custom,0.004528,0.015496,0.009496,42.397659,39.558817
1,Implicit lib,0.004496,0.015826,0.009521,2.242506,10.681962


# Implicit ALS on full train

In [63]:
R_full = csr_matrix((train["rating"].values, (train["user_idx"].values, train["item_idx"].values)), shape=(n_users, n_items))
print(f"R_full: shape={R_full.shape}, non-zeros={R_full.nnz:,}")

R_full: shape=(692133, 308122), non-zeros=3,230,426


In [64]:
als_full = implicit.als.AlternatingLeastSquares(factors=64, regularization=0.1, alpha=20.0,
                                                iterations=15, random_state=42, num_threads=0,)
t0 = time.time()
als_full.fit(R_full)
print(f"Full training time: {(time.time() - t0):.1f} s")

100%|██████████| 15/15 [00:26<00:00,  1.75s/it]

Full training time: 26.6 s


In [65]:
val_user_idx = np.array([user_to_idx[u] for u in val_user_ids])
metrics_full = evaluate_implicit(
    model=als_full,
    user_idx_local=val_user_idx,      
    R=R_full,
    gt_padded=gt_padded,
    gt_sizes=gt_sizes,
    K=10,
)

In [66]:
print("implicit ALS (full train) on val:")
for k, v in metrics_full.items():
    print(f"{k}: {v:.6f}")

implicit ALS (full train) on val:
precision@10: 0.003039
recall@10: 0.017118
ndcg@10: 0.009278
predict time (s): 68.917353
